In [25]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [5]:
df = pd.read_csv("Tweets.csv", encoding="latin-1")

In [6]:
df = df.iloc[1:].reset_index(drop=True)

df.columns = ['target', 'ids', 'date', 'flag', 'user', 'text']

In [7]:
df.columns

Index(['target', 'ids', 'date', 'flag', 'user', 'text'], dtype='str')

In [8]:
df.head(1)

,target,ids,date,flag,user,text
0,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...


In [9]:
df = df.drop(columns=["flag","user","date","ids"])

In [10]:
df.head(1)

,target,text
0,0,@Kenichan I dived many times for the ball. Man...


In [11]:
df['target'] = df['target'].replace({
    0: 0,
    2: 1,
    4: 2
})

In [12]:
df['target'].value_counts()

target
2    800000
0    799998
Name: count, dtype: int64

In [13]:
df = df.dropna(subset=['text', 'target'])

In [14]:
df = df.drop_duplicates(subset=['text'])

In [15]:
df.isnull().sum()
df.duplicated(subset=['text']).sum()

np.int64(0)

In [16]:
def clean_tweet(tweet):
    tweet = str(tweet)
    tweet = re.sub(r'http\S+|www\S+', '', tweet)
    tweet = re.sub(r'@\w+', 'USER', tweet)
    tweet = re.sub(r'#(\w+)', r'\1', tweet)
    tweet = re.sub(r'\s+', ' ', tweet).strip()

    return tweet

df['clean_text'] = df['text'].apply(clean_tweet)

In [17]:
X = df['clean_text']
y = df['target']

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [19]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    lowercase=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

In [20]:
print("Training:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Testing:", X_test_tfidf.shape)

Training: (1107024, 50000)
Validation: (237220, 50000)
Testing: (237220, 50000)


In [21]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

nb_predictions = nb_model.predict(X_val_tfidf)

print("Naive Bayes Accuracy:",
      accuracy_score(y_val, nb_predictions))

print(classification_report(y_val, nb_predictions))

Naive Bayes Accuracy: 0.7976224601635612
              precision    recall  f1-score   support

           0       0.79      0.80      0.80    118527
           2       0.80      0.79      0.80    118693

    accuracy                           0.80    237220
   macro avg       0.80      0.80      0.80    237220
weighted avg       0.80      0.80      0.80    237220



In [22]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

lr_predictions = lr_model.predict(X_val_tfidf)

print("Logistic Regression Accuracy:",
      accuracy_score(y_val, lr_predictions))

print(classification_report(y_val, lr_predictions))

Logistic Regression Accuracy: 0.817565972514965
              precision    recall  f1-score   support

           0       0.82      0.81      0.82    118527
           2       0.81      0.83      0.82    118693

    accuracy                           0.82    237220
   macro avg       0.82      0.82      0.82    237220
weighted avg       0.82      0.82      0.82    237220



In [23]:
svm_model = LinearSVC(max_iter=2000)

svm_model.fit(X_train_tfidf, y_train)

svm_predictions = svm_model.predict(X_val_tfidf)

print("SVM Accuracy:",
      accuracy_score(y_val, svm_predictions))

print(classification_report(y_val, svm_predictions))

SVM Accuracy: 0.8151252002360678
              precision    recall  f1-score   support

           0       0.82      0.80      0.81    118527
           2       0.81      0.83      0.82    118693

    accuracy                           0.82    237220
   macro avg       0.82      0.82      0.82    237220
weighted avg       0.82      0.82      0.82    237220



In [ ]:
#BERT